In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import json

In [2]:
df = pd.read_csv("../data/scrape/honestdoor_property_details.csv")

# Remove junk columns
df = df.loc[:, ~df.columns.str.contains("^Unnamed")]

display(df.head())

,Property URL,id,assessmentClass,zoning,bathroomsTotal,bathroomsTotalEst,bedroomsTotal,bedroomsTotalEst,livingArea,livingAreaEst,...,houseStyle,livingAreaUnits,basement,unparsedAddress,neighbourhoodName,closeDate,closePrice,location,privateListing,listings
0,https://www.honestdoor.com/property/11647-124-...,599258d50e39c7eaa471f1c9,Condo Building,RA8,NaN,4.0,NaN,4.0,NaN,2068.0,...,NaN,NaN,NaN,11647 124 STREET NW,Inglewood,2026-04-30T00:00:00.000Z,1100000,"{""lat"": 53.5682487, ""lon"": -113.5354309}",NaN,[]
1,https://www.honestdoor.com/property/15116-116-...,599258d40e39c7eaa471bb51,Commercial,IH,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,15116 116A AVENUE NW,Garside Industrial,2026-04-30T00:00:00.000Z,475000,"{""lat"": 53.568769, ""lon"": -113.581799}",NaN,[]
2,https://www.honestdoor.com/property/10749-181-...,599258d70e39c7eaa472fee5,Commercial,IM,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,10749 181 STREET NW,Wilson Industrial,2026-04-30T00:00:00.000Z,340000,"{""lat"": 53.554474, ""lon"": -113.63276}",NaN,[]
3,https://www.honestdoor.com/property/18925-ston...,599258df0e39c7eaa4761fe0,Commercial,DC2,NaN,NaN,NaN,NaN,3644.0,NaN,...,NaN,NaN,NaN,18925 STONY PLAIN ROAD NW,Place La Rue,2026-04-30T00:00:00.000Z,16713670,"{""lat"": 53.5395317, ""lon"": -113.6511154}",NaN,[]
4,https://www.honestdoor.com/property/18920-100-...,599258d30e39c7eaa4712314,Commercial,DC2,NaN,NaN,NaN,NaN,26823.0,NaN,...,NaN,NaN,NaN,18920 100 AVENUE NW,Place La Rue,2026-04-30T00:00:00.000Z,7997998,"{""lat"": 53.5376511, ""lon"": -113.6511765}",NaN,[]


# Fixing PrivateListing

In [3]:
import json
import pandas as pd
import numpy as np

PL_col = [
    "bathroomsTotal",
    "bedroomsTotal",
    "houseStyle",
    "livingArea",
    "lotSizeArea",
    "yearBuiltActual",
    "basement",

    # additionalInfo
    "fireplaces",
    "custom_text",
    "type_of_house",
    "front_facing_exposure",

    # additionalInfoExtra
    "parking",
    "exterior",
    "rlooring",
    "roof_type",
    "foundation",
    "road_access",
    "cooling_type",
    "heating_type",
    "title_storage",
    "heating_source",
    "site_influences",
    "construction_type",
    "parking_plan_type",
    "condo_fee_includes",
    "fireplace_fuel_type",
    "amenities_features_continued",
]

# Add PL_ columns if missing
for col in PL_col:
    pl_col = f"PL_{col}"
    if pl_col not in df.columns:
        df[pl_col] = None

pl_cols = [c for c in df.columns if c.startswith("PL_")]
df[pl_cols] = df[pl_cols].astype("object")


def load_json_value(value):
    if pd.isna(value) or value == "":
        return {}

    if isinstance(value, dict):
        return value

    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return {}

    return {}


def get_inspect_value(data, col):
    if not isinstance(data, dict):
        return None

    value = data.get(col)

    if value is None:
        value = (data.get("additionalInfo") or {}).get(col)

    if value is None:
        value = (data.get("additionalInfoExtra") or {}).get(col)

    if isinstance(value, (list, dict)):
        value = json.dumps(value, ensure_ascii=False)

    return value


for idx in df.index:
    # Row 3369 uses inspect.json
    if idx == 3369:
        with open("./inspect.json", "r", encoding="utf-8") as f:
            inspect_data = json.load(f)
    else:
        inspect_data = load_json_value(df.at[idx, "privateListing"])

    for col in PL_col:
        inspect_value = get_inspect_value(inspect_data, col)
        df.at[idx, f"PL_{col}"] = inspect_value

    if idx == 3369:
        for col in PL_col:
            df_value = df.loc[idx].get(f"PL_{col}")
            print(f"{col}: {df_value}")

bathroomsTotal: 3.1
bedroomsTotal: 5
houseStyle: HOUSE
livingArea: 2200
lotSizeArea: 653
yearBuiltActual: 1988
basement: FINISHED
fireplaces: wood fireplace w/ mantel
custom_text: garage - heated, indoor, insulated, oversized?
Built-in mudroom
type_of_house: 2 Storey
front_facing_exposure: E
parking: ["PK4 - Double Garage Attached"]
exterior: ["EX3 - Brick", "EX14 - Vinyl"]
rlooring: ["FL10 - Hardwood", "FL5 - Ceramic Tile", "FL2 - Carpet"]
roof_type: ["RT1 - Asphalt Shingles"]
foundation: ["FD3 - Concrete"]
road_access: ["RD6 - Paved"]
cooling_type: ["C1 - Central Air Conditioning"]
heating_type: ["HT3 - Forced Air-1"]
title_storage: []
heating_source: ["HS5 - Natural Gas"]
site_influences: ["SI13 - Fenced", "SI21 - Landscaped", "SI32 - Playground Nearby", "SI45 - Schools"]
construction_type: ["FRAME - Wood Frame"]
parking_plan_type: []
condo_fee_includes: []
fireplace_fuel_type: ["FF4 - Wood"]
amenities_features_continued: ["AF1 - Air Conditioner", "AF13 - Deck", "AF19 - Fire Pit", "

In [4]:
# last 30 col of row 3369
print(df.loc[3369, df.columns[-25:]])

PL_houseStyle                                                                  HOUSE
PL_livingArea                                                                   2200
PL_lotSizeArea                                                                   653
PL_yearBuiltActual                                                              1988
PL_basement                                                                 FINISHED
PL_fireplaces                                               wood fireplace w/ mantel
PL_custom_text                     garage - heated, indoor, insulated, oversized?...
PL_type_of_house                                                            2 Storey
PL_front_facing_exposure                                                           E
PL_parking                                          ["PK4 - Double Garage Attached"]
PL_exterior                                          ["EX3 - Brick", "EX14 - Vinyl"]
PL_rlooring                        ["FL10 - Hardwood", "FL5 - Cer

# Fix Listings

In [5]:
import pandas as pd

DROP_KEYS = {
    "HOAFee", "alternateURLVideoLink", "amperage", "analyticsClick",
    "bathrooms", "description", "garage", "leaseTerms",
    "liveStreamEventURL", "moreInformationLink", "numBathrooms",
    "numBathroomsPlus", "numBedrooms", "numBedroomsPlus",
    "numFireplaces", "numGarageSpaces", "parkCostMonthly",
    "virtualTourUrl", "yearBuilt", "zoning", "zoningDescription",
    "zoningType", "sqft", "sqftRange"
}

new_rows = []
L_col = []

for index, row in df.iterrows():
    listing_data = load_json_value(row["listings"])

    if not listing_data:
        new_rows.append({})
        continue

    listing_details = listing_data[0].get("details") or {}

    # Build L_col from the first valid row only
    if not L_col:
        L_col = [
            f"L_{col}"
            for col in listing_details.keys()
            if col not in DROP_KEYS
        ]

    row_values = {}

    for col in listing_details.keys():
        if col in DROP_KEYS:
            continue
        
        row_values[f"L_{col}"] = listing_details[col]

    new_rows.append(row_values)

# Create all columns at once
listing_df = pd.DataFrame(new_rows, index=df.index)

# Ensure every column in L_col exists
listing_df = listing_df.reindex(columns=L_col)

df = pd.concat([df, listing_df], axis=1)

# Aggregation

In [6]:
def choose(*values):
    for v in values:
        if pd.notna(v):
            return v
    return None

df["bathroomsCount"] = [
    choose(a, b, c)
    for a, b, c in zip(
        df["bathroomsTotal"],
        df["PL_bathroomsTotal"], 
        df["bathroomsTotalEst"],
    )
]

df["bedroomsCount"] = [
    choose(a, b, c)
    for a, b, c in zip(
        df["bedroomsTotal"],
        df["PL_bedroomsTotal"],
        df["bedroomsTotalEst"],
    )
]

df["houseStyle"] = [
    choose(a, b, c)
    for a, b, c in zip(
        df["houseStyle"],
        df["PL_type_of_house"],
        df["L_style"],
    )
]

df["livingArea"] = [
    choose(a, b, c)
    for a, b, c in zip(
        df["livingArea"],
        df["PL_livingArea"],
        df["livingAreaEst"],
    )
]

df["lotSizeArea"] = [
    choose(a, b, c)
    for a, b, c in zip(
        df["lotSizeArea"],
        df["PL_lotSizeArea"],
        df["lotSizeAreaEst"],
    )
]

df["yearBuiltActual"] = [
    choose(a, b)
    for a, b in zip(
        df["yearBuiltActual"],
        df["PL_yearBuiltActual"],
    )
]

df["basement"] = [
    choose(a, b)
    for a, b in zip(
        df["basement"],
        df["PL_basement"],
    )
]

In [7]:
def extract_lat_lon(x):
    if pd.isna(x):
        return pd.Series([None, None])

    try:
        obj = json.loads(x)
        return pd.Series([
            obj.get("lat"),
            obj.get("lon")
        ])
    except:
        return pd.Series([None, None])

df[["lat", "lon"]] = df["location"].apply(extract_lat_lon)

In [8]:
def fireplace_to_binary(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().lower()

    if x in ["yes"]:
        return 1

    return 0

df["fireplace"] = df["fireplace"].apply(fireplace_to_binary)

In [10]:
invalid_rows = df[
    ~df["Property URL"]
    .astype(str)
    .str.strip()
    .str.contains("https://", regex=False, na=False)
]

print(f"Found {len(invalid_rows)} invalid rows")

for idx, row in invalid_rows.iterrows():
    print(f"\nRow {idx}:")
    print("propertyURL =", repr(row["Property URL"]))

Found 0 invalid rows


In [ ]:
clean_df = pd.DataFrame({
    "propertyUrl": df["Property URL"],
    "assessmentClass": df["assessmentClass"],
    "zoning": df["zoning"],
    "bathroomsCount": df["bathroomsCount"],
    "bedroomsCount": df["bedroomsCount"],
    "livingArea": df["livingArea"],
    "lotSizeArea": df["lotSizeArea"],
    "yearBuilt": df["yearBuiltActual"],
    "fireplace": df["fireplace"],
    "garage": df["garageSpaces"],
    "houseStyle": df["houseStyle"],
    "basement": df["basement"],
    "address": df["unparsedAddress"],
    "neighbourhoodName": df["neighbourhoodName"],
    "closeDate": df["closeDate"],
    "price": df["closePrice"],
    "lat": df["lat"],
    "lon": df["lon"],
})

# add L_col
clean_df = pd.concat(
    [
        clean_df,
        df[L_col],
    ],
    axis=1,
)

display(clean_df.head())

,propertyUrl,assessmentClass,zoning,bathroomsCount,bedroomsCount,livingArea,lotSizeArea,yearBuilt,fireplace,garage,...,L_patio,L_propertyType,L_roofMaterial,L_sewer,L_storageType,L_style,L_swimmingPool,L_viewType,L_waterSource,L_waterfront
0,https://www.honestdoor.com/property/11647-124-...,Condo Building,RA8,4.0,4.0,2068.0,697.0,1962.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.honestdoor.com/property/15116-116-...,Commercial,IH,NaN,NaN,NaN,NaN,1977.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.honestdoor.com/property/10749-181-...,Commercial,IM,NaN,NaN,NaN,NaN,2002.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.honestdoor.com/property/18925-ston...,Commercial,DC2,NaN,NaN,3644.0,0.0,2005.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.honestdoor.com/property/18920-100-...,Commercial,DC2,NaN,NaN,26823.0,0.0,2013.0,NaN,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [12]:
missing_pct = clean_df.isna().mean() * 100

print("Missing percentage by column:")
for col, pct in missing_pct.sort_values(ascending=False).items():
    print(f"{col}: {pct:.2f}%")

Missing percentage by column:
L_energyCertification: 100.00%
L_certificationLevel: 100.00%
L_constructionStatus: 100.00%
L_commonElementsIncluded: 100.00%
L_constructionStyleSplitLevel: 100.00%
L_businessSubType: 100.00%
L_numDrivewaySpaces: 100.00%
L_numKitchens: 100.00%
L_numKitchensPlus: 100.00%
L_numRooms: 100.00%
L_numRoomsPlus: 100.00%
L_viewType: 100.00%
L_driveway: 100.00%
L_exteriorConstruction2: 100.00%
L_den: 100.00%
L_centralVac: 100.00%
L_centralAirConditioning: 100.00%
L_ceilingType: 100.00%
L_sewer: 100.00%
L_storageType: 100.00%
L_swimmingPool: 100.00%
L_waterfront: 100.00%
L_landAccessType: 100.00%
L_landDisposition: 100.00%
L_laundryLevel: 100.00%
L_greenPropertyInformationStatement: 100.00%
L_handicappedEquipped: 100.00%
L_fireProtection: 100.00%
L_landscapeFeatures: 100.00%
L_loadingType: 100.00%
L_farmType: 100.00%
L_familyRoom: 100.00%
L_landSewer: 99.91%
L_furnished: 99.91%
L_patio: 99.90%
L_airConditioning: 99.90%
L_waterSource: 99.88%
L_energuideRating: 96.90%


In [13]:
# Drop columns with more than 50% missing values except for "basement" Col
missing_cols = [
    col for col in clean_df.columns
    if missing_pct[col] > 50 and col != "basement"
]
print("missing_cols:", missing_cols)
clean_df = clean_df.drop(columns=missing_cols, errors="ignore")

missing_cols: ['fireplace', 'L_airConditioning', 'L_businessSubType', 'L_businessType', 'L_ceilingType', 'L_centralAirConditioning', 'L_centralVac', 'L_certificationLevel', 'L_commonElementsIncluded', 'L_constructionStatus', 'L_constructionStyleSplitLevel', 'L_den', 'L_driveway', 'L_elevator', 'L_energuideRating', 'L_energyCertification', 'L_exteriorConstruction2', 'L_familyRoom', 'L_farmType', 'L_fireProtection', 'L_furnished', 'L_greenPropertyInformationStatement', 'L_handicappedEquipped', 'L_landAccessType', 'L_landDisposition', 'L_landSewer', 'L_landscapeFeatures', 'L_laundryLevel', 'L_loadingType', 'L_numDrivewaySpaces', 'L_numKitchens', 'L_numKitchensPlus', 'L_numParkingSpaces', 'L_numRooms', 'L_numRoomsPlus', 'L_patio', 'L_sewer', 'L_storageType', 'L_swimmingPool', 'L_viewType', 'L_waterSource', 'L_waterfront']


In [14]:
# drop redundant col L_style, L_livingAreaMeasurement, L_propertyType
clean_df = clean_df.drop(columns=["L_style", "L_livingAreaMeasurement", "L_propertyType"], errors="ignore")

In [15]:
display(clean_df.head())

,propertyUrl,assessmentClass,zoning,bathroomsCount,bedroomsCount,livingArea,lotSizeArea,yearBuilt,garage,houseStyle,...,lat,lon,L_basement1,L_basement2,L_exteriorConstruction1,L_extras,L_flooringType,L_foundationType,L_heating,L_roofMaterial
0,https://www.honestdoor.com/property/11647-124-...,Condo Building,RA8,4.0,4.0,2068.0,697.0,1962.0,0.0,NaN,...,53.568249,-113.535431,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.honestdoor.com/property/15116-116-...,Commercial,IH,NaN,NaN,NaN,NaN,1977.0,0.0,NaN,...,53.568769,-113.581799,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.honestdoor.com/property/10749-181-...,Commercial,IM,NaN,NaN,NaN,NaN,2002.0,0.0,NaN,...,53.554474,-113.632760,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.honestdoor.com/property/18925-ston...,Commercial,DC2,NaN,NaN,3644.0,0.0,2005.0,0.0,NaN,...,53.539532,-113.651115,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.honestdoor.com/property/18920-100-...,Commercial,DC2,NaN,NaN,26823.0,0.0,2013.0,0.0,NaN,...,53.537651,-113.651177,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# save clean_df to csv
clean_df.to_csv("../data/clean/honestdoor_property_details_clean.csv", index=False)